# Phase 5: Extract Error Cases for Relabeling

This notebook loads validation predictions and exports false positives and false negatives
for each class so we can prioritize relabeling and retraining.

In [ ]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
DATA_ROOT = Path('/root/separate_volume')
if not DATA_ROOT.exists():
    DATA_ROOT = REPO_ROOT

# Update this to the run you want to analyze.
# Leave empty to auto-select the latest run.
RUN_ID = ''

runs_root = DATA_ROOT / 'training/artifacts/runs'
if RUN_ID:
    run_dir = runs_root / RUN_ID
else:
    run_dirs = [p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith('run_')]
    if not run_dirs:
        raise FileNotFoundError(f'No run_* directories found under {runs_root}')
    run_dir = max(run_dirs, key=lambda p: p.stat().st_mtime)
    RUN_ID = run_dir.name

REPORT_DIR = run_dir / 'reports'
INPUT_CSV = REPORT_DIR / 'phase5_sbert_val_predictions_with_probs.csv'
OUTPUT_DIR = REPORT_DIR / 'error_cases'

REPORT_DIR, INPUT_CSV

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Missing predictions file: {INPUT_CSV}')

df = pd.read_csv(INPUT_CSV)
if 'y_true' not in df.columns:
    raise ValueError('Expected y_true column in predictions file')

# Prefer tuned predictions when present.
pred_col = 'y_pred_tuned' if 'y_pred_tuned' in df.columns else 'y_pred'
if pred_col not in df.columns:
    raise ValueError('Expected y_pred or y_pred_tuned in predictions file')

df['y_true'] = df['y_true'].astype(str).str.upper()
df[pred_col] = df[pred_col].astype(str).str.upper()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classes = ['DISINFO', 'HATE', 'NORMAL']
summary = {}

for label in classes:
    fp = df[(df[pred_col] == label) & (df['y_true'] != label)].copy()
    fn = df[(df['y_true'] == label) & (df[pred_col] != label)].copy()
    fp_path = OUTPUT_DIR / f'false_positive_{label.lower()}.csv'
    fn_path = OUTPUT_DIR / f'false_negative_{label.lower()}.csv'
    fp.to_csv(fp_path, index=False, encoding='utf-8')
    fn.to_csv(fn_path, index=False, encoding='utf-8')
    summary[label] = {
        'false_positive': len(fp),
        'false_negative': len(fn),
        'fp_path': str(fp_path),
        'fn_path': str(fn_path),
    }

# Combined error file
errors = df[df['y_true'] != df[pred_col]].copy()
errors['error_type'] = errors.apply(
    lambda row: f"FP_{row[pred_col]}" if row['y_true'] != row[pred_col] and row[pred_col] in classes else 'ERROR',
    axis=1,
)
errors.to_csv(OUTPUT_DIR / 'error_cases_all.csv', index=False, encoding='utf-8')

summary